# NB03 — Limpieza y etiquetado

Consume `metadata/` y `data_cruda/`; produce el parquet maestro y `limpieza_log.csv`, según marco v2 §6.

In [ ]:
# CONFIG único — S2 D11/D12/D15.
import os, hashlib
from pathlib import Path
import pandas as pd
BASE_DIR=Path(os.environ.get('DERM_IAPP_BASE','/content/derm_iapp'))
CONFIG={'BASE_DIR':BASE_DIR,'FULL_DATASET':os.environ.get('DERM_IAPP_FULL_DATASET','1')=='1','SEMILLA':2026,'IDS_CASI_DUPLICADOS':tuple(x for x in os.environ.get('DERM_IAPP_IDS_EXCLUIR','ISIC_0034511,ISIC_0034619,ISIC_0068690').split(',') if x)}
CLASES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC','UNK']

def cargar_csvs():
    m=BASE_DIR/'metadata'; tm=pd.read_csv(m/'ISIC_2019_Training_Metadata.csv',dtype=str); tg=pd.read_csv(m/'ISIC_2019_Training_GroundTruth.csv'); em=pd.read_csv(m/'ISIC_2019_Test_Metadata.csv',dtype=str); eg=pd.read_csv(m/'ISIC_2019_Test_GroundTruth.csv')
    assert list(tm.columns)==['image','age_approx','anatom_site_general','lesion_id','sex']; assert list(em.columns)==['image','age_approx','anatom_site_general','sex']; assert list(tg.columns)==['image']+CLASES; assert list(eg.columns)==['image']+CLASES+['score_weight','validation_weight']
    return tm,tg,em,eg.drop(columns=['score_weight','validation_weight'])
def derivar(meta,gt,origen):
    assert (gt[CLASES].sum(axis=1)==1).all(),'one-hot inválido'; x=meta.merge(gt,on='image',validate='one_to_one').rename(columns={'image':'image_id'}); x['clase_9']=x[CLASES].idxmax(axis=1); x['es_bcc']=x.clase_9.eq('BCC'); x['origen']=origen
    x['lesion_id']=x.get('lesion_id',pd.Series([None]*len(x))).replace(['','nan'],None); x['edad']=pd.to_numeric(x.age_approx,errors='coerce').replace(0,pd.NA).astype('Int64'); x['sexo']=x.sex.replace(['','nan'],None); x['sitio']=x.anatom_site_general.replace(['','nan'],None); carpeta='ISIC_2019_Training_Input' if origen=='train' else 'ISIC_2019_Test_Input'; x['ruta']=x.image_id.map(lambda z:f'{origen}/{carpeta}/{z}.jpg'); return x[['image_id','origen','clase_9','es_bcc','lesion_id','edad','sexo','sitio','ruta']]
def hashes(x):
    filas=[]
    for r in x.itertuples():
        p=BASE_DIR/'data_cruda'/r.ruta; h=hashlib.sha256()
        with open(p,'rb') as f:
            for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
        filas.append((r.image_id,h.hexdigest(),p.stat().st_size))
    return pd.DataFrame(filas,columns=['image_id','sha256','size_bytes'])
def main():
    tm,tg,em,eg=cargar_csvs(); bruto=pd.concat([derivar(tm,tg,'train'),derivar(em,eg,'test')],ignore_index=True)
    for o in ['train','test']:
        carpeta='ISIC_2019_Training_Input' if o=='train' else 'ISIC_2019_Test_Input'; assert len(list((BASE_DIR/'data_cruda'/o/carpeta).glob('*.jpg')))==sum(bruto.origen.eq(o))
    d=bruto.merge(hashes(bruto),on='image_id'); log=[]
    def baja(mask,motivo,detalle=''):
        nonlocal d; q=d[mask]; log.extend((r.image_id,r.origen,r.clase_9,motivo,detalle) for r in q.itertuples()); d=d[~mask].copy()
    baja(d.clase_9.eq('UNK'),'UNK_sin_etiqueta'); borrar=[]
    for _,g in d[d.duplicated('sha256',keep=False)].groupby('sha256'): borrar.extend(g[g.image_id.ne(g.image_id.min())].index)
    baja(d.index.isin(borrar),'duplicado_eliminado','conserva image_id menor'); baja(d.image_id.isin(CONFIG['IDS_CASI_DUPLICADOS']),'excluido_casi_duplicado'); assert not d.sha256.duplicated().any() and not d.clase_9.eq('UNK').any()
    d['destino_asignado']='pendiente'; cols=['image_id','origen','clase_9','es_bcc','lesion_id','edad','sexo','sitio','ruta','sha256','destino_asignado']; d=d[cols].sort_values('image_id').reset_index(drop=True); d.to_parquet(BASE_DIR/'dataset_etiquetado.parquet',index=False); pd.DataFrame(log,columns=['image_id','origen','clase_9','motivo','detalle']).sort_values('image_id').to_csv(BASE_DIR/'limpieza_log.csv',index=False)
    if CONFIG['FULL_DATASET']: assert (len(d),d.es_bcc.sum(),len(d)-d.es_bcc.sum(),len(log))==(31469,4298,27171,2100)
    return d
# main se ejecuta al final, tras definir la verificación D17.

# Extensión S2: cache SHA-256 y verificación de extracción.
from pandas.testing import assert_frame_equal
def verificar_extraccion(df):
    for origen in ('train','test'):
        interno='ISIC_2019_Training_Input' if origen=='train' else 'ISIC_2019_Test_Input'
        reales=len(list((BASE_DIR/'data_cruda'/origen/interno).glob('*.jpg')))
        assert reales==int((df.origen==origen).sum()), f'JPG/CSV no coincide: {origen}'
        if CONFIG['FULL_DATASET']: assert reales==(25331 if origen=='train' else 8238)
def calcular_hashes(df):
    partes=[]
    for origen,g in df.groupby('origen',sort=True):
        cp=BASE_DIR/f'hashes_cache_{origen}.csv'
        viejo=pd.read_csv(cp,dtype={'image_id':str,'sha256':str}) if cp.exists() else pd.DataFrame(columns=['image_id','sha256','size_bytes'])
        cache={r.image_id:(r.sha256,int(r.size_bytes)) for r in viejo.itertuples()}
        filas=[]
        for i,r in enumerate(g.sort_values('image_id').itertuples(),1):
            archivo=BASE_DIR/'data_cruda'/r.ruta; tamaño=archivo.stat().st_size
            if r.image_id in cache and cache[r.image_id][1]==tamaño: h=cache[r.image_id][0]
            else:
                sha=hashlib.sha256()
                with open(archivo,'rb') as f:
                    for bloque in iter(lambda:f.read(1024*1024),b''): sha.update(bloque)
                h=sha.hexdigest()
            filas.append((r.image_id,h,tamaño))
            if i%2000==0: print(f'{origen}: {i}')
        c=pd.DataFrame(filas,columns=['image_id','sha256','size_bytes']); tmp=cp.with_suffix('.tmp'); c.to_csv(tmp,index=False); tmp.replace(cp); partes.append(c)
    return pd.concat(partes,ignore_index=True)

hashes=calcular_hashes


In [ ]:
# D17 marco v2: segunda derivación completa y comparación estricta del parquet maestro.
import io

def limpiar_etiquetar(bruto,hashes_df):
    d=bruto.merge(hashes_df,on='image_id'); log=[]
    def baja(mask,motivo,detalle=''):
        nonlocal d; q=d[mask]; log.extend((r.image_id,r.origen,r.clase_9,motivo,detalle) for r in q.itertuples()); d=d[~mask].copy()
    baja(d.clase_9.eq('UNK'),'UNK_sin_etiqueta'); borrar=[]
    for _,g in d[d.duplicated('sha256',keep=False)].groupby('sha256'): borrar.extend(g[g.image_id.ne(g.image_id.min())].index)
    baja(d.index.isin(borrar),'duplicado_eliminado','conserva image_id menor'); baja(d.image_id.isin(CONFIG['IDS_CASI_DUPLICADOS']),'excluido_casi_duplicado'); assert not d.sha256.duplicated().any() and not d.clase_9.eq('UNK').any()
    d['destino_asignado']='pendiente'; cols=['image_id','origen','clase_9','es_bcc','lesion_id','edad','sexo','sitio','ruta','sha256','destino_asignado']
    return normalizar_dtypes(d[cols].sort_values('image_id').reset_index(drop=True)),log

def normalizar_dtypes(df):
    """Canoniza nulos y columnas textuales para comparar el round-trip Parquet."""
    x=df.copy(); texto=['image_id','origen','clase_9','lesion_id','sexo','sitio','ruta','sha256','destino_asignado']
    for columna in texto: x[columna]=x[columna].astype(object).where(x[columna].notna(),None)
    return x

def verificar_determinismo(df_escrito):
    # D17 del marco v2: la re-derivación usa CSV, caché SHA-256 y limpiezas completas.
    tm,tg,em,eg=cargar_csvs(); bruto=pd.concat([derivar(tm,tg,'train'),derivar(em,eg,'test')],ignore_index=True)
    verificar_extraccion(bruto); rederivado,_=limpiar_etiquetar(bruto,calcular_hashes(bruto)); df_escrito=normalizar_dtypes(df_escrito); rederivado=normalizar_dtypes(rederivado)
    try:
        assert_frame_equal(df_escrito,rederivado,check_dtype=True,check_exact=True)
        escrito,segunda=io.BytesIO(),io.BytesIO(); df_escrito.to_parquet(escrito,index=False); rederivado.to_parquet(segunda,index=False)
        assert hashlib.sha256(escrito.getvalue()).hexdigest()==hashlib.sha256(segunda.getvalue()).hexdigest(), 'bytes parquet distintos'
    except AssertionError as e:
        raise AssertionError(f'D17 incumplida: la re-derivación no es determinista ({e})') from e

def main():
    tm,tg,em,eg=cargar_csvs(); bruto=pd.concat([derivar(tm,tg,'train'),derivar(em,eg,'test')],ignore_index=True)
    verificar_extraccion(bruto); d,log=limpiar_etiquetar(bruto,calcular_hashes(bruto))
    ruta=BASE_DIR/'dataset_etiquetado.parquet'; d.to_parquet(ruta,index=False); pd.DataFrame(log,columns=['image_id','origen','clase_9','motivo','detalle']).sort_values('image_id').to_csv(BASE_DIR/'limpieza_log.csv',index=False)
    verificar_determinismo(pd.read_parquet(ruta))
    if CONFIG['FULL_DATASET']: assert (len(d),d.es_bcc.sum(),len(d)-d.es_bcc.sum(),len(log))==(31469,4298,27171,2100)
    return d

if __name__=='__main__': main()
